In [ ]:
%pip install ipywidgets plotly anywidget

In [ ]:
import numpy as np
import scipy as sp
from ipywidgets import interactive_output, Layout, GridspecLayout
import ipywidgets as widgets
from IPython.display import display, HTML
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Parameters
fc = 5.1
fs = 256.3
N = 2**12
Vrms = 1
A = Vrms * np.sqrt(2)
border = ''
footer_height = '32px'
include_fc_lobes = 0
fig_min_width = '900px'

# Defaults
bits_default = 16
n0_default   = -200
d_default    = 1

# Globals
kwargs      = {}
fig         = None
widget_dict = None

# Dark theme for widgets, to match the plotly_dark figure

DARK_CSS = """
<style>
.jupyter-widgets, .widget-label, .widget-html-content, .widget-readout {
    color: #e0e0e0 !important;
}
.widget-box, .widget-hbox, .widget-vbox, .widget-gridbox,
.jp-OutputArea-output.jupyter-widgets-view, .cell-output-ipywidget-background {
    background-color: #1e1e1e !important;
}
.widget-readout {
    background-color: #2b2b2b !important;
}
.widget-hslider .slider-container, .ui-slider {
    background-color: #3a3a3a !important;
}
.ui-slider-handle {
    background-color: #4e9af1 !important;
    border-color: #4e9af1 !important;
}
input[type="range"], input[type="checkbox"] {
    accent-color: #4e9af1;
}
</style>
"""
display(HTML(DARK_CSS))

# Computation layer (logic unchanged from original)

def calc_metrics(**kw):
    SINAD_t = 20 * np.log10(np.sqrt(np.mean(kw['x']**2)) / np.sqrt(np.var(kw['q_e'])))
    ENOB_t  = (SINAD_t - 1.76) / 6.02
    peak_mag = 10**(kw['peak'][~np.isnan(kw['peak'])] / 10)
    NAD_mag  = 10**(kw['NAD'][~np.isnan(kw['NAD'])]   / 10)
    Ps   = fs/N * np.sum(peak_mag)
    Pnad = fs/N * np.sum(NAD_mag)
    Px   = fs/N * (np.sum(peak_mag) + np.sum(NAD_mag))
    SINAD_f = 10 * np.log10(Ps / Pnad)
    ENOB_f  = (SINAD_f - 1.76) / 6.02
    return {**kw, 'SINAD_t': SINAD_t, 'ENOB_t': ENOB_t,
            'SINAD_f': SINAD_f, 'ENOB_f': ENOB_f,
            'Ps': Ps, 'Pnad': Pnad, 'Px': Px}

def time_domain(**kw):
    d   = kw['d']**4
    t   = np.arange(N) / fs
    x   = A * np.sin(2 * np.pi * fc * t)
    a   = [1, -0.1, -0.1]
    x_d = a[0]*x + d*a[1]*x**2 + d*a[2]*x**3 + d*0.01*x**11
    s   = np.sqrt(np.mean(x**2)) / np.sqrt(np.mean(x_d**2))
    x_d = s * x_d
    x_n = np.sqrt(10**(kw['n0']/10)) * np.random.randn(N)
    x_q = x_n + A*np.round(x_d/A * (2**(kw['bits']-1)-1)) / (2**(kw['bits']-1)-1)
    q_e = x - x_q
    q   = A / (2**(kw['bits']-1))
    q_n = (q / np.sqrt(12)) * np.random.randn(N)
    return {**kw, 't': t, 'x': x, 'x_q': x_q, 'q_e': q_e, 'q_n': q_n,
            'levels': 2**kw['bits'] - 1}

def frequency_domain(**kw):
    f     = sp.fft.fftfreq(N, 1/fs)[:N//2]
    w     = sp.signal.windows.kaiser(N, beta=14)
    c_w   = abs(sum(w))
    c_d   = np.sqrt(sum(w**2))
    ssfft = lambda x: 2/c_w * sp.fft.fft(x*w)[:N//2]
    psd   = lambda x: 1/fs * np.abs(c_w*(x/np.sqrt(2))/c_d)**2
    to_db = lambda x: 10*np.log10(psd(ssfft(x)))
    return {**kw, 'f': f,
            'X':   to_db(kw['x']),
            'X_q': to_db(kw['x_q']),
            'Q_e': to_db(kw['q_e']),
            'Q_n': to_db(kw['q_n']),
            'w':   w}

def split_spectrum(**kw):
    dX   = np.diff(kw['X'])
    zc   = np.where(np.diff(np.sign(dX)))[0]
    ll   = include_fc_lobes
    ctr  = np.argmin(np.abs(fc - kw['f'][zc]))
    p0, p1 = zc[ctr-(2*ll+1)], zc[ctr+(2*ll+1)]
    pmask  = np.array([False]*p0 + [True]*(p1-p0+1) + [False]*(len(kw['X'])-p1-1))
    dX_q   = np.diff(kw['X_q'][:p0])
    if dX_q[0] >= 0:
        dmask = np.array([False]*len(kw['X']))
    else:
        dc_end = np.nonzero(np.diff(np.sign(dX_q)))[0][0] + 2 - 1
        dmask  = np.array([True]*dc_end + [False]*(len(kw['X'])-dc_end))
    peak = np.where(pmask,           kw['X_q'], np.nan)
    NAD  = np.where(~pmask & ~dmask, kw['X_q'], np.nan)
    DC   = np.where(dmask,           kw['X_q'], np.nan)
    return {**kw, 'peak': peak, 'NAD': NAD, 'DC': DC}

def compute_row(base_kw):
    F_M     = 18
    f_keys  = ['X', 'X_q', 'Q_e', 'Q_n']
    results = [frequency_domain(**time_domain(**base_kw)) for _ in range(F_M)]
    stacked = {k: np.stack([r[k] for r in results]) for k in f_keys}
    avg     = {k: stacked[k].mean(axis=0) for k in f_keys}
    merged  = {**results[0], **avg}
    merged  = split_spectrum(**merged)
    merged  = calc_metrics(**merged)
    return merged

# Widget setup

C = {'x':  '#4e9af1', 'xq': '#f0883e', 'qe': '#4ec94e', 'qn': '#e05252',
     'X':  '#4e9af1', 'pk': '#f0883e', 'nd': '#e05252', 'dc': '#9b72d0', 'Qn': '#8ec994'}

def _build_fig():
    global fig
    specs = [[{"type":"xy"}, {"type":"xy"}, {"type":"table"}],
             [{"type":"xy"}, {"type":"xy"}, {"type":"table"}]]
    fig = go.FigureWidget(make_subplots(
        rows=2, cols=3, specs=specs,
        column_widths=[0.38, 0.38, 0.24],
        vertical_spacing=0.12, horizontal_spacing=0.08,
    ))

    def add_row(row, SL):
        # time domain — traces off+0..off+3
        fig.add_trace(go.Scatter(name='x (Ideal)',        line=dict(color=C['x']),                                    showlegend=SL), row=row, col=1)
        fig.add_trace(go.Scatter(name='x_q (Quantized)',  line=dict(color=C['xq']),                                   showlegend=SL), row=row, col=1)
        fig.add_trace(go.Scatter(name='q_e (Quant Err)',  mode='markers', marker=dict(size=2, color=C['qe']), visible=False, showlegend=SL), row=row, col=1)
        fig.add_trace(go.Scatter(name='q_n (Ideal Noise)',mode='markers', marker=dict(size=2, color=C['qn']), visible=False, showlegend=SL), row=row, col=1)
        # freq domain — traces off+4..off+8
        fig.add_trace(go.Scatter(name='X (Ideal)',   line=dict(color=C['X']),                showlegend=SL), row=row, col=2)
        fig.add_trace(go.Scatter(name='X_q Signal',  line=dict(dash='dash', color=C['pk']), showlegend=SL), row=row, col=2)
        fig.add_trace(go.Scatter(name='X_q NAD',     line=dict(dash='dash', color=C['nd']), showlegend=SL), row=row, col=2)
        fig.add_trace(go.Scatter(name='X_q DC',      line=dict(dash='dash', color=C['dc']), showlegend=SL), row=row, col=2)
        fig.add_trace(go.Scatter(name='Ideal q_n',   line=dict(color=C['Qn']),  visible=False, showlegend=SL), row=row, col=2)
        # stats table — trace off+9
        ncols = 4 if SL else 3
        hdr   = ['', 'dBW', 'mW', 'dmW'][:ncols]
        empty = [['Ps','Pnad','Px']] + [['','','']] * (ncols - 1)
        fig.add_trace(go.Table(
            header=dict(values=hdr, fill_color='#2b2b2b', font=dict(color='white', size=11), height=24),
            cells=dict(values=empty, fill_color='#1a1a1a', font=dict(color='white', size=11), height=22),
        ), row=row, col=3)

    add_row(1, True)   # data[0..9]
    add_row(2, False)  # data[10..19]

    fig.update_layout(
        template='plotly_dark',
        autosize=True,
        height=560,
        margin=dict(l=55, r=15, t=40, b=45),
        legend=dict(orientation='h', yanchor='bottom', y=1.01, xanchor='left', x=0, font=dict(size=10)),
    )
    fig.update_xaxes(showticklabels=False, row=1, col=1)
    fig.update_xaxes(showticklabels=False, row=1, col=2)
    fig.update_xaxes(title_text='Time [s]',       row=2, col=1)
    fig.update_xaxes(title_text='Frequency [Hz]', row=2, col=2)
    # automargin alone can't prevent the overlap: it only grows the OUTER
    # figure margins, it never renegotiates the domain split between
    # subplot columns (that gap is fixed at horizontal_spacing, a constant
    # fraction of the figure width, so its pixel size always shrinks with
    # the figure). The real guarantee comes from fig_min_width below, which
    # stops the figure from ever getting narrow enough for that gap to be
    # smaller than the tick label + title footprint. automargin is kept
    # since it still helps at the figure's outer edges.
    fig.update_yaxes(title_text='Amplitude (V)',  col=1, automargin=True)
    fig.update_yaxes(title_text='PSD [dBW/Hz]',   col=2, automargin=True)

    with widget_dict['figure']:
        display(fig)

def widget_setup():
    global widget_dict
    widget_dict = {
        'bits':  widgets.IntSlider(min=2, max=16, step=1, value=bits_default, description='Bits'),
        'n0':    widgets.FloatSlider(min=-200, max=0, step=1, value=n0_default, description='AWGN (dB)'),
        'd':     widgets.FloatSlider(min=0, max=1, step=.1, value=d_default, description='Distortion'),
        'q_e_en': widgets.Checkbox(value=False, description='', indent=False,
                                    layout=Layout(width='auto', justify_content='center', align_items='center', border=border)),
        'q_n_en': widgets.Checkbox(value=False, description='', indent=False,
                                    layout=Layout(width='auto', justify_content='center', align_items='center', border=border)),
        'td_label': widgets.HTML(value="<div style='font-size:24px;text-align:center;'>Time Domain</div>",
                                  layout=Layout(height='100%', width='100%', border=border)),
        'fd_label': widgets.HTML(value="<div style='font-size:24px;text-align:center;'>Frequency Domain</div>",
                                  layout=Layout(height='100%', width='100%', border=border)),
        # min_width is the actual guard against the y-axis-label overlap: it
        # stops the figure from being squeezed narrower than the point where
        # the column gap can no longer fit the tick labels/title (see
        # fig_min_width and the comment in _build_fig). Below that width the
        # grid scrolls horizontally instead of squishing the plot further.
        'figure': widgets.Output(layout=Layout(width='100%', height='100%', min_width=fig_min_width, border=border)),
        'x0lim':  widgets.FloatSlider(min=1, max=np.ceil(fc*N/fs), step=1, value=4, description='XLim (T)',
                                       layout=Layout(height=footer_height, width='100%', border=border)),
        'y0lim':  widgets.FloatRangeSlider(min=-2, max=2, step=0.1, value=(-2, 2), description='YLim (V)',
                                            layout=Layout(height=footer_height, width='100%', border=border)),
        'x1lim':  widgets.FloatSlider(min=np.round(fc), max=fs/2, step=1, value=fs/4, description='XLim (Hz)',
                                       layout=Layout(height=footer_height, width='100%', border=border)),
        'y1lim':  widgets.FloatRangeSlider(min=-200, max=20, step=0.1, value=(-180, 20), description='YLim (dB)',
                                            layout=Layout(height=footer_height, width='100%', border=border)),
    }

    header = [[widget_dict['bits'], widget_dict['n0'], widget_dict['d']]]

    enable_grid = GridspecLayout(1, 4)
    enable_grid[0, 0] = widgets.HBox([widget_dict['q_e_en'], widgets.Label(value='Show Q_e')],
                                      layout=Layout(justify_content='center', width='100%', border=border))
    enable_grid[0, 1] = widgets.HBox([widget_dict['q_n_en'], widgets.Label(value='Show Q_n')],
                                      layout=Layout(justify_content='center', width='100%', border=border))

    ax_grids  = 5
    ax_ratio  = 8
    labelgrid = GridspecLayout(1, ax_grids*ax_ratio, height='44px', layout=Layout(overflow='visible'))
    labelgrid[0, 1:2*ax_ratio]             = widget_dict['td_label']
    labelgrid[0, 2*ax_ratio+1:4*ax_ratio]  = widget_dict['fd_label']

    x_slider_grid = GridspecLayout(1, ax_grids, layout=Layout(overflow='visible'))
    x_slider_grid[0, 0:-1] = widgets.HBox([widget_dict['x0lim'], widget_dict['x1lim']],
                                            layout=Layout(justify_content='flex-end', width='100%', border=border))
    y_slider_grid = GridspecLayout(1, ax_grids, layout=Layout(overflow='visible'))
    y_slider_grid[0, 0:-1] = widgets.HBox([widget_dict['y0lim'], widget_dict['y1lim']],
                                            layout=Layout(justify_content='center', width='100%', border=border))

    grid = GridspecLayout(20, 3, grid_gap='0px',
                           layout=widgets.Layout(align_content='stretch', justify_content='space-around', border=border))

    def fill_grid(row_start, grid_items):
        for i, row in enumerate(grid_items):
            for j, item in enumerate(row):
                item.layout = Layout(justify_content='center', width='100%', border=border)
                w = int(grid.n_columns / len(row))
                grid[i + row_start, j*w:(j+1)*w] = item

    fill_grid(0, header)
    grid[1, :]    = enable_grid
    grid[2, :]    = labelgrid
    grid[3:-1, :] = widget_dict['figure']
    grid[-1:, :]  = widgets.VBox([x_slider_grid, y_slider_grid])

    # Give each conceptual section its own fixed row height instead of 20 equal
    # fractions — otherwise the figure's 560px forces every row (incl. the
    # title and footer-slider rows) to the same cramped ~35px, which is too
    # short for their content and causes a vertical scrollbar to appear.
    # Must be set AFTER all grid[...] assignments above: GridspecLayout's
    # __setitem__ calls _update_layout(), which resets grid_template_rows
    # back to 'repeat(n_rows, 1fr)' on every cell assignment.
    grid.layout.grid_template_rows = '40px 40px 44px ' + '35px ' * 16 + '70px'

    _build_fig()
    return grid

# Draw callback

def draw_plot(**ui):
    global kwargs
    kwargs = {**kwargs, **ui}

    r0 = compute_row(dict(kwargs))
    r0['Vpp']  = np.ptp(r0['x_q'])
    r0['Vrms'] = np.sqrt(np.mean(r0['x_q']**2))

    for key in ['Vpp', 'Vrms', 'Px', 'Ps', 'Pnad']:
        r0['delta_' + key] = r0[key] - kwargs.get('prev_' + key, 0.0)
        kwargs['prev_' + key] = r0[key]

    r1 = compute_row({**dict(kwargs), 'bits': r0['ENOB_f'], 'n0': -300, 'd': 0.0})

    qe = ui.get('q_e_en', False)
    qn = ui.get('q_n_en', False)

    def set_row(r, off, show_delta):
        d = fig.data
        d[off+0].x = r['t'];  d[off+0].y = r['x']
        d[off+1].x = r['t'];  d[off+1].y = r['x_q']
        d[off+2].x = r['t'];  d[off+2].y = r['q_e'];  d[off+2].visible = qe
        d[off+3].x = r['t'];  d[off+3].y = r['q_n'];  d[off+3].visible = qn
        d[off+4].x = r['f'];  d[off+4].y = r['X']
        d[off+5].x = r['f'];  d[off+5].y = r['peak']
        d[off+6].x = r['f'];  d[off+6].y = r['NAD']
        d[off+7].x = r['f'];  d[off+7].y = r['DC']
        d[off+8].x = r['f'];  d[off+8].y = r['Q_n'];  d[off+8].visible = qn
        db_col  = [f"{10*np.log10(r['Ps']):.3f}",  f"{10*np.log10(r['Pnad']):.3f}",  f"{10*np.log10(r['Px']):.3f}"]
        mw_col  = [f"{r['Ps']*1e3:.3f}",            f"{r['Pnad']*1e3:.3f}",            f"{r['Px']*1e3:.3f}"]
        vals = [['Ps','Pnad','Px'], db_col, mw_col]
        if show_delta:
            vals.append([f"{r.get('delta_Ps',0)*1e3:+.3g}",
                         f"{r.get('delta_Pnad',0)*1e3:+.3g}",
                         f"{r.get('delta_Px',0)*1e3:+.3g}"])
        d[off+9].cells.values = vals

    eff  = np.log2(int(np.floor(2**r0['bits'] - 1)) + 1)
    ann0 = (f"N={r0['bits']:.3g}b ({eff:.3g} eff) | SINAD={r0['SINAD_f']:.2f} dBW | "
            f"ENOB={r0['ENOB_f']:.2f}b | Vpp={r0['Vpp']:.2f}V | Vrms={r0['Vrms']:.2f}V")
    ann1 = (f"ENOB ref: N={r0['ENOB_f']:.3g}b | SINAD={r1['SINAD_f']:.2f} dBW | ENOB={r1['ENOB_f']:.2f}b")

    with fig.batch_update():
        set_row(r0,  0, True)
        set_row(r1, 10, False)
        fig.update_xaxes(range=[0, ui['x0lim']/fc], row=1, col=1)
        fig.update_xaxes(range=[0, ui['x0lim']/fc], row=2, col=1)
        fig.update_yaxes(range=list(ui['y0lim']),   row=1, col=1)
        fig.update_yaxes(range=list(ui['y0lim']),   row=2, col=1)
        fig.update_xaxes(range=[0, ui['x1lim']],    row=1, col=2)
        fig.update_xaxes(range=[0, ui['x1lim']],    row=2, col=2)
        fig.update_yaxes(range=list(ui['y1lim']),   row=1, col=2)
        fig.update_yaxes(range=list(ui['y1lim']),   row=2, col=2)
        fig.update_layout(annotations=[
            dict(text=ann0, xref='paper', yref='paper', x=0.01, y=1.03,
                 xanchor='left', showarrow=False, font=dict(size=10, color='white')),
            dict(text=ann1, xref='paper', yref='paper', x=0.01, y=0.49,
                 xanchor='left', showarrow=False, font=dict(size=10, color='white')),
        ])

# Main

grid = widget_setup()
out  = interactive_output(draw_plot, {k: v for k, v in widget_dict.items() if k != 'figure'})
display(grid, out)
